## Cleaning the data & Extracting only necessary features for training

In [26]:
import re
import time
import json
import pandas as pd
from pathlib import Path
from datasets import load_dataset

    Loading dataset

In [2]:
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

In [3]:
data = dataset['train'].to_pandas()

In [4]:
data.columns

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='str')

    Removing all unneeded columns

In [6]:
data = data.drop(columns=['flags', 'category', 'response'])
data.head()

,instruction,intent
0,question about cancelling order {{Order Number}},cancel_order
1,i have a question about cancelling oorder {{Or...,cancel_order
2,i need help cancelling puchase {{Order Number}},cancel_order
3,I need to cancel purchase {{Order Number}},cancel_order
4,"I cannot afford this order, cancel purchase {{...",cancel_order


    Making a function to clean dataset

In [7]:
def clean_text_instructions(text: str) -> str:
    # Convert to lowercase
    text = text.lower()

    # Replace variable templates (e.g., {{Order Number}}) with a standard token
    text = re.sub(r'\{\{[\s\w]+\}\}', 'unk_placeholder', text)

    # Remove all punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Strip extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [11]:
print("Processing text data... (This might take a few seconds)")
start = time.perf_counter()
data['cleaned_text'] = data['instruction'].apply(clean_text_instructions)
end = time.perf_counter()
print(f"Text data processing completed, time taken - {end-start}")

Processing text data... (This might take a few seconds)
Text data processing completed, time taken - 0.14237759999923583


    Converting intent into categories (numeric label)

In [13]:
data['intent'] = data['intent'].astype('category')
data['label'] = data['intent'].cat.codes

    Storing a map to know and convert the categories

In [18]:
# Create a mapping dictionary to store the relationship for inference later
label_mapping = dict(enumerate(data['intent'].cat.categories))

In [19]:
data.sample(5)

,instruction,intent,cleaned_text,label
3032,how can I check the early exit fees?,check_cancellation_fee,how can i check the early exit fees,3
16147,I have paid {{Currency Symbol}}{{Refund Amount...,get_refund,i have paid unk_placeholderunk_placeholder for...,16
26373,I'm waiting for a restitution of {{Currency Sy...,track_refund,im waiting for a restitution of unk_placeholde...,26
25960,I'm waiting for a compensation of {{Currency ...,track_refund,im waiting for a compensation of unk_placehold...,26
16516,I paid {{Currency Symbol}}{{Refund Amount}} fo...,get_refund,i paid unk_placeholderunk_placeholder for this...,16


    Dropping columns that do not need any more

In [20]:
data = data.drop(columns=['instruction', 'intent'])

In [21]:
data.sample(10)

,cleaned_text,label
24918,i dont know how i can check the eta of purchas...,25
12418,i have got to see what delivery options are there,12
2830,i have got to modify my shipping address how t...,2
12881,help me seeing what delivery methods i can choose,12
5380,i do not know how i can see your payment modal...,5
16193,i have paid unk_placeholderunk_placeholder for...,16
12438,i want assistance to see what shipment methods...,12
16557,i have paid unk_placeholderunk_placeholder for...,16
14112,could you help me editing my fucking account,14
21317,i cant inform of an issue with signup,21


    Saving our cleaned & transformed data

In [24]:
# Directory path
dir_path = Path("../data_info")

# Output Filename
output_filename = "cleaned_data_for_training.csv"

# Create folder if does not exist
dir_path.mkdir(parents=True, exist_ok=True)

# File Path
file_path = dir_path/output_filename

# Converting to csv without index
data.to_csv(file_path, index=False)

print(f"Successfully generated: {output_filename} at directory {dir_path}")

Successfully generated: cleaned_data_for_training.csv at directory ..\data_info


    Saving our label mapping

In [25]:
label_mapping

{0: 'cancel_order',
 1: 'change_order',
 2: 'change_shipping_address',
 3: 'check_cancellation_fee',
 4: 'check_invoice',
 5: 'check_payment_methods',
 6: 'check_refund_policy',
 7: 'complaint',
 8: 'contact_customer_service',
 9: 'contact_human_agent',
 10: 'create_account',
 11: 'delete_account',
 12: 'delivery_options',
 13: 'delivery_period',
 14: 'edit_account',
 15: 'get_invoice',
 16: 'get_refund',
 17: 'newsletter_subscription',
 18: 'payment_issue',
 19: 'place_order',
 20: 'recover_password',
 21: 'registration_problems',
 22: 'review',
 23: 'set_up_shipping_address',
 24: 'switch_account',
 25: 'track_order',
 26: 'track_refund'}

    Saving Label Mapping as a json file

In [28]:
clean_mapping = {int(key): str(value) for key, value in label_mapping.items()}

json_file_path = dir_path / "label_mapping.json"

with open(json_file_path, "w") as f:
    json.dump(clean_mapping, f, indent=4)

print(f"Saved {json_file_path} successfully!")

Saved ..\data_info\label_mapping.json successfully!
